# [STARTER] Udaplay Project

## Part 01 - Offline RAG

In this part of the project, you'll build your VectorDB using Chroma.

The data is inside folder `project/starter/games`. Each file will become a document in the collection you'll create.
Example.:
```json
{
  "Name": "Gran Turismo",
  "Platform": "PlayStation 1",
  "Genre": "Racing",
  "Publisher": "Sony Computer Entertainment",
  "Description": "A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.",
  "YearOfRelease": 1997
}
```


### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
import os
import json
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv

In [3]:
# TODO: Create a .env file with the following variables
# OPENAI_API_KEY="YOUR_KEY"
# CHROMA_OPENAI_API_KEY="YOUR_KEY"
# TAVILY_API_KEY="YOUR_KEY"

# Add .env file to environment variables


In [4]:
# TODO: Load environment variables
# load_dotenv()

load_dotenv(".env")
required_env_vars = ["OPENAI_API_KEY", "CHROMA_OPENAI_API_KEY", "TAVILY_API_KEY"]

print("Environment variable checklist:")
all_present = True
for key in required_env_vars:
   exists = bool(os.getenv(key))
   all_present = all_present and exists
   print(f"- {key}: {'OK' if exists else 'MISSING'}")

print(f"\nAll required env vars present: {all_present}")

Environment variable checklist:
- OPENAI_API_KEY: OK
- CHROMA_OPENAI_API_KEY: OK
- TAVILY_API_KEY: OK

All required env vars present: True


### VectorDB Instance

In [5]:
import tempfile

db_path = "./chromadb"
os.makedirs(db_path, exist_ok=True)

try:
    chroma_client = chromadb.PersistentClient(path=db_path)
    print(f"Initialized ChromaDB at: {os.path.abspath(db_path)}")
except Exception:
    fallback_db_path = os.path.join(tempfile.gettempdir(), "udaplay_chromadb")
    os.makedirs(fallback_db_path, exist_ok=True)
    chroma_client = chromadb.PersistentClient(path=fallback_db_path)
    print(f"Could not open ./chromadb, switched to: {fallback_db_path}")

chroma_client

Initialized ChromaDB at: /workspace/Code/project/starter/chromadb


### Collection

In [6]:
# TODO: Pick one embedding function
# If picking something different than openai, 
# make sure you use the same when loading it
embedding_fn = embedding_functions.OpenAIEmbeddingFunction()

In [7]:
# Create a fresh collection by deleting existing one first
collection_name = "udaplay"

try:
    chroma_client.delete_collection(name=collection_name)
    print(f"Deleted existing collection: {collection_name}")
except Exception:
    pass

collection = chroma_client.create_collection(
    name=collection_name,
    embedding_function=embedding_fn
)
collection

Collection(name=udaplay)

### Add documents

In [8]:
# Make sure you have a directory "project/starter/games"
data_dir = "games"

for file_name in sorted(os.listdir(data_dir)):
    if not file_name.endswith(".json"):
        continue

    file_path = os.path.join(data_dir, file_name)
    with open(file_path, "r", encoding="utf-8") as f:
        game = json.load(f)

    # You can change what text you want to index
    content = f"[{game['Platform']}] {game['Name']} ({game['YearOfRelease']}) - {game['Description']}"

    # Use file name (like 001) as ID
    doc_id = os.path.splitext(file_name)[0]

    collection.add(
        ids=[doc_id],
        documents=[content],
        metadatas=[game]
    )

# Retrieve all records from ChromaDB and print each record as JSON
records = collection.get(include=["documents", "metadatas"])

for i, record_id in enumerate(records.get("ids", [])):
    row = {
        "id": record_id,
        "document": records["documents"][i] if records.get("documents") else None,
        "metadata": records["metadatas"][i] if records.get("metadatas") else None,
    }
    print(json.dumps(row, ensure_ascii=False, indent=2))

{
  "id": "001",
  "document": "[PlayStation 1] Gran Turismo (1997) - A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.",
  "metadata": {
    "Name": "Gran Turismo",
    "Description": "A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.",
    "YearOfRelease": 1997,
    "Publisher": "Sony Computer Entertainment",
    "Genre": "Racing",
    "Platform": "PlayStation 1"
  }
}
{
  "id": "002",
  "document": "[PlayStation 2] Grand Theft Auto: San Andreas (2004) - An expansive open-world game set in the fictional state of San Andreas, following the story of Carl 'CJ' Johnson.",
  "metadata": {
    "Platform": "PlayStation 2",
    "YearOfRelease": 2004,
    "Genre": "Action-adventure",
    "Publisher": "Rockstar Games",
    "Name": "Grand Theft Auto: San Andreas",
    "Description": "An expansive open-world game set in the fictional state of San Andreas, following the story of